In [15]:
import os
import random
import glob

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [16]:
# -----------------------------
# 1. ACH FILE GENERATOR 
# The program below generates 5 ACH files with 1000 records each
# It chooses a random value between 0 and 1 and if it is <0.5, it creates a credit transaction from a ramdon set of trancodes. 
# And if not, it creates a debit transaction from pre-defined set of random transaction codes.
# -----------------------------

ROUTING_OPTIONS = [
    "02100002",  # NY
    "12100035",  # CA
    "09100001",  # Midwest
    "06100010",  # Southeast
    "03110050",  # Pennsylvania / Mid‑Atlantic
    "07192500",  # Illinois / Great Lakes
    "11100061",  # Texas / Southwest

]

N_FILES = 5
RECORDS_PER_FILE = 1000

os.makedirs("data", exist_ok=True)

def make_entry_detail():
    # Choose routing region
    routing = random.choice(ROUTING_OPTIONS)
    check_digit = str(sum(int(d) for d in routing) % 10)
    dfi = routing + check_digit  # 9 chars

    # Account number: 17 chars, fixed
    account = str(random.randint(0, 99999999999999999)).zfill(17)

    # Inject structure:
    # - credits tend to be larger amounts
    # - certain routing regions have systematically higher/lower amounts
    is_credit = random.random() < 0.5
    if is_credit:
        transaction_code = random.choice(["20", "21","22", "23", "24"])  # credits
        base_amount = random.randint(5000, 50000)
    else:
        transaction_code = random.choice(["25","26","27","28", "29", "30", "31", "32"])  # debits
        base_amount = random.randint(50, 5000)
    if routing.startswith("12"):      # West
        base_amount = int(base_amount * 1.5)
    elif routing.startswith("09"):    # Midwest
        base_amount = int(base_amount * 0.7)
    elif routing.startswith("06"):    # Southeast
        base_amount = int(base_amount * 1.2)

    amount = max(1, min(base_amount, 9999999999))  # clamp
    amount_str = str(amount).zfill(10)             # 10 chars

    # Individual ID: 15 chars
    individual_id = str(random.randint(0, 999999999999999)).zfill(15)

    # Name: 22 chars
    individual_name = random.choice(["JOHN DOE", "JANE SMITH", "ALICE BROWN"]).ljust(22)

    # Discretionary data: 2 chars
    discretionary = "  "

    # Addenda indicator: 1 char
    addenda = "0"

    # Trace number: 15 chars
    trace = "12345678" + str(random.randint(0, 99999999)).zfill(8)

    line = (
        "6" +                 # 1
        transaction_code +    # 2
        dfi +                 # 9
        account +             # 17
        amount_str +          # 10
        individual_id +       # 15
        individual_name +     # 22
        discretionary +       # 2
        addenda +             # 1
        trace                 # 15
    )

    # Guarantee exactly 94 characters
    line = line[:94]
    return line

def write_ach_files():
    for i in range(1, N_FILES + 1):
        path = os.path.join("data", f"ach{i}.txt")
        with open(path, "w", newline="") as fh:
            # Simple file header + batch header (dummy, not parsed)
            fh.write("101 1 SIMPLE ACH FILE HEADER DUMMY LINE.........................\n")
            fh.write("5200TEST BATCH HEADER DUMMY LINE...............................\n")
            for _ in range(RECORDS_PER_FILE):
                fh.write(make_entry_detail() + "\n")

write_ach_files()
